# Scaling Law Experiment (Colab)

Run MBB synthetic data augmentation scaling law experiments on Google Colab.

**Requirements:** Colab GPU runtime (T4 or better). Go to *Runtime > Change runtime type > GPU*.

In [ ]:
# --- 1. Setup: clone repo and install deps ---
import os

REPO_URL = "https://github.com/jamesdchen/harxhar.git"  # your repo URL
REPO_DIR = "harxhar"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
!pip install -q torch transformers numpy pandas scikit-learn tqdm pyarrow numba

In [ ]:
# --- 2. Verify GPU ---
from harxhar_dl.notebook_utils import verify_gpu

verify_gpu()

In [ ]:
# --- 3. Upload your data ---
# Uncomment ONE of the options below:

# Option A: Mount Google Drive (recommended for large datasets)
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_PATH = '/content/drive/MyDrive/harxhar_data/all30min'

# Option B: Upload parquet files from local machine
# import os
# from google.colab import files
# os.makedirs('all30min', exist_ok=True)
# uploaded = files.upload()  # upload parquet files, then move them:
# for fname in uploaded:
#     os.rename(fname, os.path.join('all30min', fname))
# DATA_PATH = 'all30min'

# Option C: Data already in repo
DATA_PATH = "all30min"

In [ ]:
# --- 4. Configuration ---
import copy
import os

import pandas as pd
import torch
from harxhar_core.data import load_and_prep_data_strided

from harxhar_dl.backtest.gpu_engine_scaling import run_scaling_experiment
from harxhar_dl.config import DL_CONFIG
from harxhar_dl.notebook_utils import configure_cuda

configure_cuda()

# --- Experiment Configuration ---
SCALE_MULTIPLIERS = [0, 1, 2, 5, 10, 50]
N_REPEATS = 3
BLOCK_SIZE = 48  # one trading day of 30-min bars
TRAIN_FRAC = 0.8
RESULTS_DIR = "results_scaling_laws"
DEVICE = torch.device("cuda")

CONFIG = copy.deepcopy(DL_CONFIG)
CONFIG["data_path"] = DATA_PATH

LOADING_HPARAMS = {
    "exog_cols": "none",
    "use_transform_exog": False,
    "use_diurnal": True,
    "allow_missing": False,
    "use_winsor": False,
}

os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Device: {DEVICE}")
print(f"Results dir: {os.path.abspath(RESULTS_DIR)}")

In [ ]:
# --- 5. Load data ---
X_np, y_np, dates, baselines, features = load_and_prep_data_strided(
    LOADING_HPARAMS, CONFIG["data_path"], lag=CONFIG["model"]["context_len"]
)
print(f"Data: {X_np.shape[0]} samples, {X_np.shape[1]} features")
print(f"Target range: [{y_np.min():.4f}, {y_np.max():.4f}]")

In [ ]:
# --- 6. Chronological train/test split ---
split_idx = int(len(X_np) * TRAIN_FRAC)

X_train, y_train = X_np[:split_idx], y_np[:split_idx]
X_test, y_test = X_np[split_idx:], y_np[split_idx:]
baselines_test = baselines[split_idx:]
dates_test = dates.iloc[split_idx:]

print(f"Train: {len(X_train)} samples ({TRAIN_FRAC:.0%})")
print(f"Test:  {len(X_test)} samples ({1 - TRAIN_FRAC:.0%})")
print(f"Test date range: {dates_test.iloc[0]} \u2192 {dates_test.iloc[-1]}")

In [ ]:
# --- 7. Run scaling experiments ---
# Results saved incrementally for fault tolerance (Colab can disconnect)
all_results = []
csv_path = os.path.join(RESULTS_DIR, "scaling_results.csv")

# Resume from previous partial run if the CSV exists
if os.path.exists(csv_path):
    prev_df = pd.read_csv(csv_path)
    done_keys = set(zip(prev_df["multiplier"], prev_df["repeat"], strict=False))
    all_results = prev_df.to_dict("records")
    print(f"Resuming: {len(done_keys)} runs already completed")
else:
    done_keys = set()

for mult in SCALE_MULTIPLIERS:
    for rep in range(N_REPEATS):
        if (mult, rep) in done_keys:
            print(f"Skipping multiplier={mult}, repeat={rep} (already done)")
            continue

        seed = mult * 1000 + rep
        print(f"\n{'=' * 60}")
        print(f"Multiplier={mult}, Repeat={rep}, Seed={seed}")
        print(f"{'=' * 60}")

        result = run_scaling_experiment(
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
            baselines_test=baselines_test,
            model_config=CONFIG["model"],
            train_config=CONFIG["train"],
            multiplier=mult,
            block_size=BLOCK_SIZE,
            seed=seed,
            device=DEVICE,
        )
        result["repeat"] = rep
        all_results.append(result)

        # Save incrementally for fault tolerance
        pd.DataFrame(all_results).drop(columns=["epoch_losses"], errors="ignore").to_csv(csv_path, index=False)
        print(f"  \u2192 QLIKE={result['qlike']:.6f}, n_windows={result['n_train_windows']}")

print(f"\nAll results saved to {csv_path}")

In [ ]:
# --- 8. Summary ---
summary = pd.DataFrame(all_results).drop(columns=["epoch_losses"], errors="ignore")
print(
    summary.groupby("multiplier")
    .agg(
        qlike_mean=("qlike", "mean"),
        qlike_std=("qlike", "std"),
        mse_mean=("mse", "mean"),
        mae_mean=("mae", "mean"),
        n_windows=("n_train_windows", "first"),
    )
    .reset_index()
    .to_string(index=False)
)

In [ ]:
# --- 9. Download results ---
from harxhar_dl.notebook_utils import download_results

download_results(csv_path)